In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from misc import model_config

In [2]:
main_model_config = (
    model_config.query("main")
    .drop(columns="main")
    .rename(columns={k: f"model_{k}" for k in model_config.columns})
)

new_name = {
    "powermoe": "PowerMoE",
    "llamamoe": "LLaMA-MoE-v1",
    "olmoe": "OLMoE",
    "switch": "SwitchTransformers",
    "llamamoe2": "LLaMA-MoE-v2",
    "jetmoe": "JetMoE",
    "openmoe": "OpenMoE",
    "minicpm": "MiniCPM-MoE",
    "qwen": "Qwen1.5-MoE",
    "deepseek2": "DeepSeek-V2-Lite",
    "deepseek": "DeepSeekMoE",
    "xverse": "XVERSE-MoE",
    "qwen3": "Qwen3",
    "yuan": "Yuan2.0",
    "phi": "Phi-3.5-MoE",
    "grin": "GRIN-MoE",
    "mixtral": "Mixtral-8x7B",
    "jamba": "Jamba-Mini",
    "nllb": "NLLB-MoE",
    "qwen2": "Qwen2",
}

model_colors = {
    key: px.colors.qualitative.Dark24[i] for i, key in enumerate(main_model_config.index.values)
}

seg_lens = (4, 16, 64, 256)
seg_len_colors = {key: px.colors.qualitative.Plotly[i] for i, key in enumerate(seg_lens)}
main_model_config

,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn
key,,,,,,,,
powermoe,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
llamamoe,LLaMA-MoE-v1-3.5B,LL1,causal,6.74,32,16,4,eager
olmoe,OLMoE-1B-7B-0125,OL,causal,6.92,16,64,8,flash_attention_2
switch,SwitchTransformers-Base-128,ST,seq2seq,7.42,24,128,1,eager
llamamoe2,LLaMA-MoE-v2-3.8B,LL2,causal,8.03,32,8,2,flash_attention_2
jetmoe,JetMoE-8B,JT,causal,8.52,24,8,2,flash_attention_2
openmoe,OpenMoE-8B,OP,causal,11.86,24,32,2,eager
minicpm,MiniCPM-MoE-8x2B,MC,causal,13.87,40,8,2,flash_attention_2
qwen,Qwen1.5-MoE-A2.7B,QW1,causal,14.32,24,60,4,flash_attention_2


In [3]:
def make_abbr(df):
    return (
        f"{df["model_abbr"]}{"d" if df["is_decoder"] else "e"}"
        if df["model_type"] == "seq2seq"
        else df["model_abbr"]
    )

In [ ]:
root_dir = Path("../output/srp_mpq")

dfs = {
    p.stem: pd.merge(pd.read_parquet(p), main_model_config, left_on="model", right_index=True)
    for p in root_dir.glob("*.parquet")
}

for df in dfs.values():
    df["model"] = df["model"].astype(model_config.index.dtype)

sorted_model_keys = (
    dfs["mg"].query("is_decoder and seg_len == 16").sort_values("best_f1", ascending=False)["model"]
)

dfs["mg"].pivot(index=["model", "is_decoder"], columns="seg_len", values=["best_f1", "best_m"])

best_f1                                  best_m  \
seg_len                    4         16        64        256       4     
model     is_decoder                                                     
powermoe  True        0.671950  0.550020  0.502823  0.482419  1.102595   
llamamoe  True        0.559473  0.453734  0.416232  0.406340  1.032373   
olmoe     True        0.641103  0.501977  0.446895  0.417123  0.993255   
switch    False       0.415518  0.189954  0.120389  0.097115  3.813264   
          True        0.417487  0.189678  0.115876       NaN  3.790567   
llamamoe2 True        0.837831  0.792673  0.777501  0.769355  1.113048   
jetmoe    True        0.601751  0.474037  0.426424  0.409788  1.092109   
openmoe   True        0.452470  0.282345  0.213235  0.183837  3.420180   
minicpm   True        0.625431  0.489046  0.437109  0.417172  1.106584   
qwen      True        0.473497  0.307055  0.222958  0.184875  3.223887   
deepseek2 True        0.529854  0.376965  0.303567  0.265276  0.837046   
deepseek  True        0.519856  0.367044  0.290904  0.252248  0.823225   
xverse    True        0.528562  0.377844  0.303323  0.265986  0.811417   
qwen3     True        0.684355  0.556722  0.495527  0.457042  1.001224   
yuan      True        0.708408  0.632207  0.609548  0.597632  0.958669   
phi       True        0.648380  0.516567  0.461993  0.433749  1.010547   
grin      True        0.637708  0.504681  0.450426  0.421622  0.995675   
mixtral   True        0.624885  0.492742  0.443175  0.422735  1.114235   
jamba     True        0.528314  0.377591  0.308152  0.275505  0.867542   
nllb      False       0.441105  0.246970  0.188444  0.160217  3.534071   
          True        0.453914  0.307490  0.256655  0.216489  3.406126   
qwen2     True        0.504599  0.368088  0.303090  0.274434  0.822677   

                                                    
seg_len                    16        64        256  
model     is_decoder                                
powermoe  True        1.389000  1.481697  1.611714  
llamamoe  True        2.385689  2.930385  3.480427  
olmoe     True        1.046252  1.185790  1.200150  
switch    False       1.940485  2.462093  1.709783  
          True        1.975091  2.482284       NaN  
llamamoe2 True        1.034039  1.059062  1.047726  
jetmoe    True        2.258621  2.704934  3.197739  
openmoe   True        1.548527  2.551589  2.345842  
minicpm   True        2.183844  2.581551  2.924181  
qwen      True        1.831509  2.054841  2.798467  
deepseek2 True        1.268557  1.959818  2.112996  
deepseek  True        2.281528  1.893916  2.171978  
xverse    True        1.216886  1.863937  1.870730  
qwen3     True        1.079037  1.112598  1.096174  
yuan      True        0.871374  0.907204  0.897233  
phi       True        1.133049  1.339387  1.317601  
grin      True        1.102532  1.292702  1.344227  
mixtral   True        2.177395  2.299254  2.790300  
jamba     True        1.448926  2.571776  2.782281  
nllb      False       3.255738  1.533978  1.805128  
          True        1.374839  1.297228  1.508917  
qwen2     True        2.588546  2.465476  2.537721

In [5]:
dfs["mg"].pivot(
    index=["model", "is_decoder"], columns="seg_len", values=["ci_lb", "ci_ub"]
).swaplevel(0, 1, axis=1).sort_index(axis=1).sort_values((16, "ci_lb"), ascending=False)

seg_len                    4                   16                  64   \
                         ci_lb     ci_ub     ci_lb     ci_ub     ci_lb   
model     is_decoder                                                     
llamamoe2 True        0.837534  0.838115  0.792261  0.793071  0.777051   
yuan      True        0.708126  0.708671  0.631796  0.632584  0.609079   
qwen3     True        0.683943  0.684764  0.556108  0.557366  0.494757   
powermoe  True        0.671716  0.672182  0.549659  0.550378  0.502383   
phi       True        0.647826  0.648972  0.515730  0.517479  0.461042   
grin      True        0.637189  0.638209  0.503867  0.505465  0.449477   
olmoe     True        0.640697  0.641547  0.501284  0.502698  0.446126   
mixtral   True        0.624738  0.625040  0.492605  0.492877  0.443007   
minicpm   True        0.625261  0.625580  0.488910  0.489176  0.436993   
jetmoe    True        0.601599  0.601900  0.473916  0.474150  0.426318   
llamamoe  True        0.559289  0.559638  0.453643  0.453828  0.416146   
xverse    True        0.528221  0.528892  0.377484  0.378205  0.303005   
jamba     True        0.528073  0.528548  0.377309  0.377846  0.307895   
deepseek2 True        0.529568  0.530149  0.376619  0.377316  0.303245   
qwen2     True        0.504442  0.504769  0.368019  0.368154  0.303013   
deepseek  True        0.519569  0.520162  0.366807  0.367269  0.290572   
nllb      True        0.453729  0.454091  0.306998  0.307964  0.256094   
qwen      True        0.473383  0.473597  0.306823  0.307259  0.222662   
openmoe   True        0.452260  0.452662  0.281842  0.282825  0.212780   
nllb      False       0.440974  0.441225  0.246715  0.247212  0.187919   
switch    False       0.415400  0.415639  0.189612  0.190304  0.120026   
          True        0.417376  0.417605  0.189287  0.190078  0.115500   

seg_len                              256            
                         ci_ub     ci_lb     ci_ub  
model     is_decoder                                
llamamoe2 True        0.777946  0.768856  0.769832  
yuan      True        0.609966  0.597112  0.598104  
qwen3     True        0.496280  0.456094  0.457983  
powermoe  True        0.503289  0.481906  0.482944  
phi       True        0.462973  0.432646  0.434938  
grin      True        0.451346  0.420478  0.422693  
olmoe     True        0.447723  0.416213  0.418053  
mixtral   True        0.443334  0.422564  0.422896  
minicpm   True        0.437220  0.417064  0.417281  
jetmoe    True        0.426533  0.409690  0.409879  
llamamoe  True        0.416319  0.406268  0.406406  
xverse    True        0.303647  0.265645  0.266343  
jamba     True        0.308398  0.275188  0.275800  
deepseek2 True        0.303893  0.264886  0.265619  
qwen2     True        0.303163  0.274346  0.274522  
deepseek  True        0.291214  0.251922  0.252588  
nllb      True        0.257201  0.215899  0.217009  
qwen      True        0.223207  0.184586  0.185133  
openmoe   True        0.213670  0.183279  0.184346  
nllb      False       0.188945  0.159680  0.160736  
switch    False       0.120768  0.096613  0.097647  
          True        0.116272       NaN       NaN

In [6]:
dfs["mg"].assign(
    ci_dist=np.maximum(
        dfs["mg"]["ci_ub"] - dfs["mg"]["best_f1"], dfs["mg"]["best_f1"] - dfs["mg"]["ci_lb"]
    )
).pivot(index=["model", "is_decoder"], columns="seg_len", values="ci_dist")

seg_len                    4         16        64        256
model     is_decoder                                        
powermoe  True        0.000234  0.000361  0.000466  0.000525
llamamoe  True        0.000185  0.000094  0.000087  0.000072
olmoe     True        0.000443  0.000721  0.000828  0.000930
switch    False       0.000120  0.000350  0.000379  0.000532
          True        0.000118  0.000400  0.000395       NaN
llamamoe2 True        0.000297  0.000412  0.000450  0.000499
jetmoe    True        0.000152  0.000121  0.000110  0.000099
openmoe   True        0.000211  0.000503  0.000454  0.000558
minicpm   True        0.000170  0.000136  0.000116  0.000109
qwen      True        0.000115  0.000231  0.000297  0.000289
deepseek2 True        0.000295  0.000351  0.000326  0.000390
deepseek  True        0.000306  0.000236  0.000332  0.000340
xverse    True        0.000341  0.000361  0.000324  0.000357
qwen3     True        0.000412  0.000644  0.000770  0.000948
yuan      True        0.000282  0.000411  0.000468  0.000520
phi       True        0.000592  0.000912  0.000980  0.001189
grin      True        0.000519  0.000814  0.000949  0.001144
mixtral   True        0.000155  0.000137  0.000169  0.000172
jamba     True        0.000241  0.000282  0.000257  0.000317
nllb      False       0.000130  0.000255  0.000525  0.000537
          True        0.000184  0.000493  0.000561  0.000589
qwen2     True        0.000169  0.000069  0.000077  0.000088

In [ ]:
mldf = pd.merge(
    dfs["mg"],
    pd.read_parquet("../output/loss.parquet")
    .groupby("model", as_index=False, observed=True)[["loss"]]
    .mean(),
)

mldf

,model,is_decoder,seg_len,act_r,best_f1,ci_lb,ci_ub,best_m,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn,loss
0,powermoe,True,4,0.200000,0.671950,0.671716,0.672182,1.102595,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2,1.915114
1,powermoe,True,16,0.200000,0.550020,0.549659,0.550378,1.389000,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2,1.915114
2,powermoe,True,64,0.200000,0.502823,0.502383,0.503289,1.481697,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2,1.915114
3,powermoe,True,256,0.200000,0.482419,0.481906,0.482944,1.611714,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2,1.915114
4,llamamoe,True,4,0.250000,0.559473,0.559289,0.559638,1.032373,LLaMA-MoE-v1-3.5B,LL1,causal,6.74,32,16,4,eager,1.941863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,nllb,True,256,0.015625,0.216489,0.215899,0.217009,1.508917,NLLB-MoE-54B,NL,seq2seq,54.50,48,128,2,eager,2.591369
83,qwen2,True,4,0.125000,0.504599,0.504442,0.504769,0.822677,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2,1.866344
84,qwen2,True,16,0.125000,0.368088,0.368019,0.368154,2.588546,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2,1.866344
85,qwen2,True,64,0.125000,0.303090,0.303013,0.303163,2.465476,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2,1.866344


In [ ]:
fig = make_subplots(rows=1, cols=len(seg_lens), shared_xaxes="all", horizontal_spacing=0.01)

text_pos = [
    {
        "powermoe": "middle right",
        "llamamoe": "top center",
        "olmoe": "middle right",
        "switch": ["middle left", "middle right"],
        "llamamoe2": "bottom center",
        "jetmoe": "top center",
        "openmoe": "middle right",
        "minicpm": "middle right",
        "qwen": "bottom center",
        "deepseek2": "middle right",
        "deepseek": "middle left",
        "xverse": "bottom right",
        "qwen3": "middle right",
        "yuan": "bottom center",
        "phi": "bottom center",
        "grin": "middle left",
        "mixtral": "top center",
        "jamba": "top center",
        "nllb": ["middle left", "middle left"],
        "qwen2": "bottom center",
    },
    {
        "powermoe": "top center",
        "llamamoe": "middle left",
        "olmoe": "bottom center",
        "switch": ["bottom center", "top center"],
        "llamamoe2": "bottom center",
        "jetmoe": "middle left",
        "openmoe": "middle left",
        "minicpm": "bottom center",
        "qwen": "bottom center",
        "deepseek2": "middle right",
        "deepseek": "bottom center",
        "xverse": "bottom center",
        "qwen3": "bottom center",
        "yuan": "bottom center",
        "phi": "top center",
        "grin": "middle left",
        "mixtral": "middle right",
        "jamba": "top center",
        "nllb": ["bottom center", "bottom center"],
        "qwen2": "middle left",
    },
    {
        "powermoe": "top center",
        "llamamoe": "middle left",
        "olmoe": "bottom center",
        "switch": ["middle right", "middle left"],
        "llamamoe2": "bottom center",
        "jetmoe": "middle left",
        "openmoe": "bottom center",
        "minicpm": "middle right",
        "qwen": "middle left",
        "deepseek2": "middle right",
        "deepseek": "middle left",
        "xverse": "bottom center",
        "qwen3": "bottom center",
        "yuan": "bottom center",
        "phi": "middle right",
        "grin": "middle left",
        "mixtral": "middle right",
        "jamba": "top center",
        "nllb": ["bottom center", "bottom center"],
        "qwen2": "bottom center",
    },
    {
        "powermoe": "top center",
        "llamamoe": "middle left",
        "olmoe": "bottom center",
        "switch": ["bottom center", "bottom center"],
        "llamamoe2": "bottom center",
        "jetmoe": "middle left",
        "openmoe": "bottom center",
        "minicpm": "middle left",
        "qwen": "bottom center",
        "deepseek2": "middle right",
        "deepseek": "middle right",
        "xverse": "bottom center",
        "qwen3": "bottom center",
        "yuan": "bottom center",
        "phi": "middle right",
        "grin": "middle left",
        "mixtral": "middle right",
        "jamba": "top center",
        "nllb": ["bottom center", "bottom center"],
        "qwen2": "middle right",
    },
]

font_size = [16, 20, 24, 28]
show_legend = True

for i, seg_len in enumerate(seg_lens):
    col = i + 1

    for j, key in enumerate(main_model_config.index.values):
        tmpdf = mldf.query(f"model == '{key}' and seg_len == {seg_len}")

        fig.add_scatter(
            x=tmpdf["best_f1"],
            y=tmpdf["best_m"],
            hoverinfo="skip",
            marker=go.scatter.Marker(
                color=model_colors[key],
                line=go.scatter.marker.Line(color="white", width=1),
                opacity=0.7,
                size=main_model_config.loc[key, "model_num_params"] ** 0.5 * 4,
            ),
            legendgroup=key,
            mode="markers+text",
            name=new_name[key],
            showlegend=show_legend,
            text=tmpdf.apply(make_abbr, axis=1),
            textfont=go.scatter.Textfont(size=font_size[0], shadow="auto"),
            textposition=text_pos[i][key],
            zorder=100 - j,
            row=1,
            col=col,
        )

        fig.update_xaxes(
            tickfont=go.layout.xaxis.Tickfont(size=font_size[1]),
            title=go.layout.xaxis.Title(
                font=go.layout.xaxis.title.Font(size=font_size[2]), text=f"SRP(E,{seg_len})"
            ),
            row=1,
            col=col,
        )

        fig.update_yaxes(range=[0.5, 4], showticklabels=col == 1, row=1, col=col)

        if col == 1:
            fig.update_yaxes(
                tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
                title=go.layout.yaxis.Title(
                    font=go.layout.yaxis.title.Font(size=font_size[2]), text="ρ(E,m)"
                ),
                row=1,
                col=col,
            )

    show_legend = False

fig.update_annotations(font=go.layout.annotation.Font(size=font_size[3]))

fig.update_layout(
    legend=go.layout.Legend(
        font=go.layout.legend.Font(size=font_size[1]),
        itemsizing="constant",
        orientation="h",
        y=-0.15,
        yanchor="top",
    ),
    margin=go.layout.Margin(l=60, r=30, t=0, b=0),
    width=2000,
    height=600,
)

plot_dir = Path("../plot")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(plot_dir / "msrp.pdf")
# fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=len(seg_lens), shared_xaxes="all", horizontal_spacing=0.01)

text_pos = [
    {
        "powermoe": "middle right",
        "llamamoe": "middle right",
        "olmoe": "bottom center",
        "llamamoe2": "bottom center",
        "jetmoe": "middle left",
        "openmoe": "bottom center",
        "minicpm": "top center",
        "qwen": "middle left",
        "deepseek2": "bottom center",
        "deepseek": "top center",
        "xverse": "bottom center",
        "qwen3": "middle right",
        "yuan": "middle right",
        "phi": "middle right",
        "grin": "middle left",
        "mixtral": "bottom center",
        "jamba": "top center",
        "qwen2": "middle left",
    },
    {
        "powermoe": "middle right",
        "llamamoe": "top center",
        "olmoe": "bottom center",
        "llamamoe2": "bottom center",
        "jetmoe": "top center",
        "openmoe": "bottom center",
        "minicpm": "top center",
        "qwen": "top center",
        "deepseek2": "middle right",
        "deepseek": "middle left",
        "xverse": "bottom center",
        "qwen3": "middle right",
        "yuan": "middle right",
        "phi": "middle right",
        "grin": "middle left",
        "mixtral": "bottom center",
        "jamba": "top center",
        "qwen2": "bottom center",
    },
    {
        "powermoe": "middle right",
        "llamamoe": "middle left",
        "olmoe": "bottom center",
        "llamamoe2": "bottom center",
        "jetmoe": "middle left",
        "openmoe": "bottom center",
        "minicpm": "top center",
        "qwen": "top center",
        "deepseek2": "middle right",
        "deepseek": "middle left",
        "xverse": "bottom center",
        "qwen3": "middle right",
        "yuan": "middle right",
        "phi": "middle right",
        "grin": "middle left",
        "mixtral": "bottom center",
        "jamba": "top center",
        "qwen2": "bottom center",
    },
    {
        "powermoe": "middle right",
        "llamamoe": "middle left",
        "olmoe": "bottom center",
        "llamamoe2": "bottom center",
        "jetmoe": "middle left",
        "openmoe": "bottom center",
        "minicpm": "top center",
        "qwen": "top center",
        "deepseek2": "bottom center",
        "deepseek": "middle left",
        "xverse": "bottom center",
        "qwen3": "middle right",
        "yuan": "middle right",
        "phi": "middle right",
        "grin": "middle left",
        "mixtral": "bottom center",
        "jamba": "top center",
        "qwen2": "middle right",
    },
]

font_size = [16, 20, 24, 28]
show_legend = True

for i, seg_len in enumerate(seg_lens):
    col = i + 1

    for j, key in enumerate(main_model_config.query("model_type == 'causal'").index.values):
        tmpdf = mldf.query(f"model == '{key}' and seg_len == {seg_len}")

        fig.add_scatter(
            x=tmpdf["best_f1"],
            y=tmpdf["loss"],
            hoverinfo="skip",
            marker=go.scatter.Marker(
                color=model_colors[key],
                line=go.scatter.marker.Line(color="white", width=1),
                opacity=0.7,
                size=main_model_config.loc[key, "model_num_params"] ** 0.5 * 4,
            ),
            legendgroup=key,
            mode="markers+text",
            name=new_name[key],
            showlegend=show_legend,
            text=tmpdf.apply(make_abbr, axis=1),
            textfont=go.scatter.Textfont(size=font_size[0], shadow="auto"),
            textposition=text_pos[i][key],
            zorder=100 - j,
            row=1,
            col=col,
        )

        fig.update_xaxes(
            tickfont=go.layout.xaxis.Tickfont(size=font_size[1]),
            title=go.layout.xaxis.Title(
                font=go.layout.xaxis.title.Font(size=font_size[2]), text=f"SRP(E,{seg_len})"
            ),
            row=1,
            col=col,
        )

        fig.update_yaxes(range=[1, 4], showticklabels=col == 1, row=1, col=col)

        if col == 1:
            fig.update_yaxes(
                tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
                title=go.layout.yaxis.Title(
                    font=go.layout.yaxis.title.Font(size=font_size[2]), text="log PPL"
                ),
                row=1,
                col=col,
            )

    show_legend = False

fig.update_annotations(font=go.layout.annotation.Font(size=font_size[3]))

fig.update_layout(
    legend=go.layout.Legend(
        font=go.layout.legend.Font(size=font_size[1]),
        itemsizing="constant",
        orientation="h",
        y=-0.15,
        yanchor="top",
    ),
    margin=go.layout.Margin(l=60, r=30, t=0, b=0),
    width=2000,
    height=600,
)

plot_dir = Path("../plot")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(plot_dir / "msrpls.pdf")
# fig.show()

In [10]:
ldf = dfs["lg"]
ldf

,model,is_decoder,layer_idx,seg_len,act_r,best_f1,ci_lb,ci_ub,best_m,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn
0,powermoe,True,0,4,0.200,0.671950,0.671716,0.672182,1.102595,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
1,powermoe,True,0,16,0.200,0.550020,0.549659,0.550378,1.389000,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
2,powermoe,True,0,64,0.200,0.502823,0.502383,0.503289,1.481697,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
3,powermoe,True,0,256,0.200,0.482419,0.481906,0.482944,1.611714,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
4,powermoe,True,1,4,0.200,0.876083,0.875967,0.876208,1.019115,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2713,qwen2,True,26,256,0.125,0.237558,0.237460,0.237661,4.469476,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
2714,qwen2,True,27,4,0.125,0.473466,0.473382,0.473552,3.224167,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
2715,qwen2,True,27,16,0.125,0.329671,0.329568,0.329762,2.588214,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
2716,qwen2,True,27,64,0.125,0.260756,0.260655,0.260856,3.258648,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2


In [ ]:
num_cols = 10
num_rows = (len(main_model_config) - 1) // num_cols + 1

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    shared_xaxes="all",
    shared_yaxes="all",
    horizontal_spacing=0.07,
    vertical_spacing=0.11,
    subplot_titles=sorted_model_keys.map(new_name).values,
    specs=[[{"secondary_y": True, "r": -0.06} for _ in range(num_cols)] for _ in range(num_rows)],
)

font_size = [16, 16, 18, 20]
show_legend = True

for i, key in enumerate(sorted_model_keys):
    if (ldf["model"] == key).sum() == 0:
        continue

    row = i // num_cols + 1
    col = i % num_cols + 1
    num_layers = model_config.loc[key, "num_layers"]

    for seg_len in seg_lens:
        tmpdf = ldf.query(f"model == '{key}' and seg_len == {seg_len}")
        if len(tmpdf) == 0:
            continue

        layer_idx = tmpdf["layer_idx"] + 1
        if model_config.loc[key, "type"] == "seq2seq":
            layer_idx += tmpdf["is_decoder"] * model_config.loc[key, "num_layers"] // 2

        f1_name = f"SRP(E,{seg_len})"
        m_name = f"ρ(E,{seg_len})"

        fig.add_scatter(
            x=layer_idx / num_layers,
            y=tmpdf["best_f1"],
            hoverinfo="skip",
            legendgroup=f1_name,
            line=go.scatter.Line(color=seg_len_colors[seg_len]),
            mode="lines",
            name=f1_name,
            showlegend=show_legend,
            row=row,
            col=col,
        )

        fig.add_scatter(
            x=layer_idx / num_layers,
            y=tmpdf["best_m"],
            customdata=layer_idx,
            hoverinfo="skip",
            legendgroup=m_name,
            line=go.scatter.Line(color=seg_len_colors[seg_len], dash="dot"),
            mode="lines",
            name=m_name,
            opacity=0.5,
            secondary_y=True,
            showlegend=show_legend,
            row=row,
            col=col,
        )

    show_legend = False

    fig.update_xaxes(
        showticklabels=True,
        tickfont=go.layout.xaxis.Tickfont(size=font_size[1]),
        tickvals=[1 / num_layers, 0.5, 1],
        ticktext=[1, num_layers // 2, num_layers],
        row=row,
        col=col,
    )

    if row == num_rows:
        fig.update_xaxes(
            title=go.layout.xaxis.Title(
                font=go.layout.xaxis.title.Font(size=font_size[2]), standoff=1, text="Layer"
            ),
            row=row,
            col=col,
        )

    fig.update_yaxes(showticklabels=col == 1, row=row, col=col, secondary_y=False)

    fig.update_yaxes(
        range=[0.5, 5.5], showticklabels=col == num_cols, row=row, col=col, secondary_y=True
    )

    if col == 1:
        fig.update_yaxes(
            tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
            title=go.layout.yaxis.Title(
                font=go.layout.yaxis.title.Font(size=font_size[2]), text="SRP(E,m)"
            ),
            row=row,
            col=col,
            secondary_y=False,
        )

    if col == num_cols:
        fig.update_yaxes(
            tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
            title=go.layout.yaxis.Title(
                font=go.layout.yaxis.title.Font(size=font_size[2]), text="ρ(E,m)"
            ),
            row=row,
            col=col,
            secondary_y=True,
        )

fig.update_annotations(font=go.layout.annotation.Font(size=font_size[3]))

fig.update_layout(
    legend=go.layout.Legend(
        font=go.layout.legend.Font(size=font_size[2]), orientation="h", x=0.47, xanchor="center"
    ),
    margin=go.layout.Margin(l=60, r=0, t=30, b=0),
    width=2000,
    height=500,
)

plot_dir = Path("../plot")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(plot_dir / "lsrp.pdf")
# fig.show()

In [12]:
sddf = pd.merge(
    dfs["mg"],
    dfs["eg"]
    .groupby(["model", "is_decoder", "seg_len"], as_index=False, observed=True)
    .aggregate(act_r_std=("act_r", "std")),
)

sddf.pivot(
    index=["model", "is_decoder"], columns="seg_len", values=["best_f1", "act_r_std"]
).sort_values(("best_f1", 16), ascending=False)

best_f1                               act_r_std  \
seg_len                    4         16        64        256       4     
model     is_decoder                                                     
llamamoe2 True        0.837831  0.792673  0.777501  0.769355  0.304180   
yuan      True        0.708408  0.632207  0.609548  0.597632  0.139459   
qwen3     True        0.684355  0.556722  0.495527  0.457042  0.043472   
powermoe  True        0.671950  0.550020  0.502823  0.482419  0.123833   
phi       True        0.648380  0.516567  0.461993  0.433749  0.040624   
grin      True        0.637708  0.504681  0.450426  0.421622  0.034751   
olmoe     True        0.641103  0.501977  0.446895  0.417123  0.052442   
mixtral   True        0.624885  0.492742  0.443175  0.422735  0.025336   
minicpm   True        0.625431  0.489046  0.437109  0.417172  0.028091   
jetmoe    True        0.601751  0.474037  0.426424  0.409788  0.009882   
llamamoe  True        0.559473  0.453734  0.416232  0.406340  0.026656   
xverse    True        0.528562  0.377844  0.303323  0.265986  0.023930   
jamba     True        0.528314  0.377591  0.308152  0.275505  0.029103   
deepseek2 True        0.529854  0.376965  0.303567  0.265276  0.019343   
qwen2     True        0.504599  0.368088  0.303090  0.274434  0.067184   
deepseek  True        0.519856  0.367044  0.290904  0.252248  0.015592   
nllb      True        0.453914  0.307490  0.256655  0.216489  0.020639   
qwen      True        0.473497  0.307055  0.222958  0.184875  0.005528   
openmoe   True        0.452470  0.282345  0.213235  0.183837  0.025904   
nllb      False       0.441105  0.246970  0.188444  0.160217  0.016669   
switch    False       0.415518  0.189954  0.120389  0.097115  0.004839   
          True        0.417487  0.189678  0.115876       NaN  0.006208   

                                                    
seg_len                    16        64        256  
model     is_decoder                                
llamamoe2 True        0.304570  0.304926  0.305216  
yuan      True        0.139444  0.139406  0.139396  
qwen3     True        0.043506  0.043435  0.043341  
powermoe  True        0.123868  0.123877  0.123903  
phi       True        0.040681  0.040713  0.040750  
grin      True        0.034767  0.034710  0.034661  
olmoe     True        0.052339  0.052037  0.051623  
mixtral   True        0.025277  0.025064  0.024862  
minicpm   True        0.027867  0.027477  0.027099  
jetmoe    True        0.009726  0.009384  0.009015  
llamamoe  True        0.026649  0.026536  0.026380  
xverse    True        0.023844  0.023538  0.023161  
jamba     True        0.029020  0.028994  0.029030  
deepseek2 True        0.019215  0.018864  0.018507  
qwen2     True        0.067224  0.067292  0.067357  
deepseek  True        0.015469  0.015164  0.014858  
nllb      True        0.020663  0.020720  0.021019  
qwen      True        0.005454  0.005302  0.005153  
openmoe   True        0.025756  0.025190  0.024510  
nllb      False       0.016697  0.016840  0.017581  
switch    False       0.004819  0.004763  0.004710  
          True        0.005854  0.006298       NaN

In [ ]:
fig = make_subplots(rows=1, cols=len(seg_lens), shared_xaxes="all", horizontal_spacing=0.01)

text_pos = [
    {
        "powermoe": "top center",
        "llamamoe": "top center",
        "olmoe": "top center",
        "switch": ["top center", "top center"],
        "llamamoe2": "top center",
        "jetmoe": "top center",
        "openmoe": "top center",
        "minicpm": "top center",
        "qwen": "top center",
        "deepseek2": "top center",
        "deepseek": "top center",
        "xverse": "top center",
        "qwen3": "top center",
        "yuan": "top center",
        "phi": "top center",
        "grin": "top center",
        "mixtral": "top center",
        "jamba": "top center",
        "nllb": ["top center", "top center"],
        "qwen2": "top center",
    },
    {
        "powermoe": "top center",
        "llamamoe": "top center",
        "olmoe": "top center",
        "switch": ["top center", "top center"],
        "llamamoe2": "top center",
        "jetmoe": "top center",
        "openmoe": "top center",
        "minicpm": "top center",
        "qwen": "top center",
        "deepseek2": "top center",
        "deepseek": "top center",
        "xverse": "top center",
        "qwen3": "top center",
        "yuan": "top center",
        "phi": "top center",
        "grin": "top center",
        "mixtral": "top center",
        "jamba": "top center",
        "nllb": ["top center", "top center"],
        "qwen2": "top center",
    },
    {
        "powermoe": "top center",
        "llamamoe": "top center",
        "olmoe": "top center",
        "switch": ["top center", "top center"],
        "llamamoe2": "top center",
        "jetmoe": "top center",
        "openmoe": "top center",
        "minicpm": "top center",
        "qwen": "top center",
        "deepseek2": "top center",
        "deepseek": "top center",
        "xverse": "top center",
        "qwen3": "top center",
        "yuan": "top center",
        "phi": "top center",
        "grin": "top center",
        "mixtral": "top center",
        "jamba": "top center",
        "nllb": ["top center", "top center"],
        "qwen2": "top center",
    },
    {
        "powermoe": "top center",
        "llamamoe": "top center",
        "olmoe": "top center",
        "switch": ["top center", "top center"],
        "llamamoe2": "top center",
        "jetmoe": "top center",
        "openmoe": "top center",
        "minicpm": "top center",
        "qwen": "top center",
        "deepseek2": "top center",
        "deepseek": "top center",
        "xverse": "top center",
        "qwen3": "top center",
        "yuan": "top center",
        "phi": "top center",
        "grin": "top center",
        "mixtral": "top center",
        "jamba": "top center",
        "nllb": ["top center", "top center"],
        "qwen2": "top center",
    },
]

font_size = [16, 20, 24, 28]
show_legend = True

for i, seg_len in enumerate(seg_lens):
    col = i + 1

    for j, key in enumerate(main_model_config.index.values):
        tmpdf = sddf.query(f"model == '{key}' and seg_len == {seg_len}")

        fig.add_scatter(
            x=tmpdf["best_f1"],
            y=tmpdf["act_r_std"],
            hoverinfo="skip",
            marker=go.scatter.Marker(
                color=model_colors[key],
                line=go.scatter.marker.Line(color="white", width=1),
                opacity=0.7,
                size=main_model_config.loc[key, "model_num_params"] ** 0.5 * 4,
            ),
            legendgroup=key,
            mode="markers+text",
            name=new_name[key],
            showlegend=show_legend,
            text=tmpdf.apply(make_abbr, axis=1),
            textfont=go.scatter.Textfont(size=font_size[0], shadow="auto"),
            textposition=text_pos[i][key],
            zorder=100 - j,
            row=1,
            col=col,
        )

        fig.update_xaxes(
            tickfont=go.layout.xaxis.Tickfont(size=font_size[1]),
            title=go.layout.xaxis.Title(
                font=go.layout.xaxis.title.Font(size=font_size[2]), text=f"SRP(E,{seg_len})"
            ),
            row=1,
            col=col,
        )

        fig.update_yaxes(range=[0, 0.35], showticklabels=col == 1, row=1, col=col)

        if col == 1:
            fig.update_yaxes(
                tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
                title=go.layout.yaxis.Title(
                    font=go.layout.yaxis.title.Font(size=font_size[2]), text="Act. Freq. SD"
                ),
                row=1,
                col=col,
            )

    show_legend = False

fig.update_annotations(font=go.layout.annotation.Font(size=font_size[3]))

fig.update_layout(
    legend=go.layout.Legend(
        font=go.layout.legend.Font(size=font_size[1]),
        itemsizing="constant",
        orientation="h",
        y=-0.15,
        yanchor="top",
    ),
    margin=go.layout.Margin(l=60, r=30, t=0, b=0),
    width=2000,
    height=600,
)

plot_dir = Path("../plot")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(plot_dir / "msrpsd.pdf")
# fig.show()

In [14]:
sample_seg_len = 16
edf = dfs["eg"].query(f"seg_len == {sample_seg_len}").drop(columns="seg_len")
edf

,model,is_decoder,layer_idx,expert_idx,act_r,best_f1,ci_lb,ci_ub,best_m,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn
1,powermoe,True,0,0,0.586158,0.754772,0.753827,0.755613,1.416378,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
5,powermoe,True,0,1,0.503463,0.687003,0.685957,0.688043,1.686383,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
9,powermoe,True,0,2,0.323470,0.544055,0.543013,0.545164,1.769881,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
13,powermoe,True,0,3,0.370275,0.589426,0.588418,0.590501,1.896490,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
17,powermoe,True,0,4,0.366373,0.578746,0.577641,0.579786,1.942623,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110573,qwen2,True,27,59,0.124398,0.328928,0.328409,0.329438,2.577562,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
110577,qwen2,True,27,60,0.132388,0.335549,0.335106,0.335973,2.690522,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
110581,qwen2,True,27,61,0.130088,0.335545,0.335082,0.336015,2.668734,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
110585,qwen2,True,27,62,0.133896,0.326095,0.325653,0.326569,2.711607,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2


In [ ]:
num_cols = 10
num_rows = (len(main_model_config) - 1) // num_cols + 1

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    shared_xaxes="all",
    shared_yaxes="all",
    horizontal_spacing=0.02,
    vertical_spacing=0.08,
    subplot_titles=sorted_model_keys.map(new_name).values,
)


def tf(x):
    return np.log(x + 1e-5) - np.log(1 - x + 1e-5)


r = np.arange(1, 10000) / 10000
rx = tf(r)
ry = 2 * r / (r + 1)
font_size = [14, 16, 18, 20]


for i, key in enumerate(sorted_model_keys):
    if (edf["model"] == key).sum() == 0:
        continue

    row = i // num_cols + 1
    col = i % num_cols + 1
    tmpdf = edf.query(f"model == '{key}'")
    num_layers = model_config.loc[key, "num_layers"]
    layer_idx = tmpdf["layer_idx"] + 1

    if model_config.loc[key, "type"] == "seq2seq":
        layer_idx += tmpdf["is_decoder"] * num_layers // 2

    fig.add_scatter(
        x=rx,
        y=ry,
        line=go.scatter.Line(color="DarkSlateGrey", dash="dot"),
        hoverinfo="skip",
        mode="lines",
        opacity=0.5,
        showlegend=False,
        row=row,
        col=col,
    )

    fig.add_vline(
        x=tf(model_config.loc[key, "top_k"] / model_config.loc[key, "num_experts"]),
        line_dash="dash",
        line_color="green",
        opacity=0.5,
        row=row,
        col=col,
    )

    fig.add_scatter(
        x=tf(tmpdf["act_r"]),
        y=tmpdf["best_f1"],
        hoverinfo="skip",
        marker=go.scatter.Marker(
            cmax=num_layers,
            cmin=1,
            color=layer_idx + 1,
            colorbar=go.scatter.marker.ColorBar(
                len=0.49,
                thickness=10,
                x=(col - 0.23) * 0.102,
                y=(num_rows - 1 - row + 1.45) * 0.54,
                tickfont=go.scatter.marker.colorbar.Tickfont(size=font_size[0]),
                tickvals=[1, num_layers // 2, num_layers],
                title=go.scatter.marker.colorbar.Title(
                    font=go.scatter.marker.colorbar.title.Font(size=font_size[1]), text="Ly."
                ),
            ),
            size=5,
        ),
        mode="markers",
        showlegend=False,
        row=row,
        col=col,
    )

    fig.update_xaxes(
        showticklabels=row == num_rows,
        tickvals=tf(np.array([0.0001, 0.01, 0.5, 0.99, 0.9999])),
        row=row,
        col=col,
    )

    if row == num_rows:
        fig.update_xaxes(
            title=go.layout.xaxis.Title(
                font=go.layout.xaxis.title.Font(size=font_size[2]), standoff=0, text="Act. Freq."
            ),
            tickfont=go.layout.xaxis.Tickfont(size=font_size[1]),
            ticktext=[".0001", ".01", ".5", ".99", ".9999"],
            row=row,
            col=col,
        )

    if col == 1:
        fig.update_yaxes(
            showticklabels=True,
            title=go.layout.yaxis.Title(
                font=go.layout.yaxis.title.Font(size=font_size[2]), text=f"SRP(e,{sample_seg_len})"
            ),
            tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
            row=row,
            col=col,
        )

fig.update_annotations(font=go.layout.annotation.Font(size=font_size[3]))
fig.update_layout(margin=go.layout.Margin(l=60, r=0, t=30, b=0), width=2000, height=500)

plot_dir = Path("../plot")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(plot_dir / "esrp.pdf")
# fig.show()